# 🔍 Tratamento e Auditoria de Dados
**UC2 — Criar e manipular dados utilizando matemática estatística**  
Prof. Cassio Ribeiro | SENAC RJ

---
**Base utilizada:** `funcionarios_naragroup_sujo.csv`  
**Objetivo:** identificar e tratar os 8 tipos de problema mais comuns em bases de dados reais.

| # | Problema | Função |
|---|---|---|
| 1 | Valores nulos | `isnull()` · `dropna()` · `fillna()` |
| 2 | Duplicatas | `duplicated()` · `drop_duplicates()` |
| 3 | Tipo de dado errado | `pd.to_numeric()` |
| 4 | Datas inconsistentes | `pd.to_datetime()` |
| 5 | Categorias sujas | `str.strip()` · `str.upper()` · `map()` |
| 6 | Espaços invisíveis | `str.strip()` |
| 7 | Valores impossíveis | filtro com `.loc` |
| 8 | Outliers extremos | IQR |


## 📦 Importações

In [2]:
import pandas as pd
import numpy as np

## 1️⃣ Carregando a base
O arquivo usa `;` como delimitador — padrão de exportação de muitos sistemas brasileiros.

In [7]:
df = pd.read_csv('funcionarios_naragroup_sujo.csv', sep=';')

print('Shape inicial:', df.shape)
df

Shape inicial: (100, 8)


,id,nome,departamento,cargo,salario,data_admissao,email,ativo
0,32,Roberto Cardoso,FINANCEIRO,Assistente,11605.79,04/09/2022,roberto.cardoso@naragroup.com,S
1,93,Amanda Silva,Financeiro,Coordenador,10032.2,09/08/2019,amanda.silva@naragroup.com,S
2,52,Jefferson Lima,T.I.,Estagiário,7387.84,2021-06-13,jefferson.lima@naragroup.com,1
3,81,Aissa Pereira,Tecnologia,Analista,10376.08,Nov-2018,aissa.pereira@naragroup.com,True
4,7,Patrícia Rocha,FINANCEIRO,Assistente,"""14859,19""",2023-12-29,patrícia.rocha@naragroup.com,SIM
...,...,...,...,...,...,...,...,...
95,42,Willian Duarte,Recursos Humanos,Analista,6032.64,Jan-2018,willian.duarte@naragroup.com,1
96,16,Rodrigo Batista,Vend.,Gerente,13975.81,16/04/2024,rodrigo.batista@naragroup.com,True
97,99,Viviane Gomes,R.H.,Diretor,9935.7,20/05/2020,viviane.gomes@naragroup.com,1
98,3,Fernanda Costa,FINANCEIRO,Coordenador,11250.05,2024-08-07,fernanda.costa@naragroup.com,True


## 🔎 Primeiro olhar: entendendo a base
Antes de qualquer tratamento, exploramos para entender o que temos.

In [8]:
# Tipos de cada coluna
display(df.dtypes)

id               int64
nome               str
departamento       str
cargo              str
salario            str
data_admissao      str
email              str
ativo              str
dtype: object

In [9]:
# Resumo completo — inclui nulos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id             100 non-null    int64
 1   nome           100 non-null    str  
 2   departamento   100 non-null    str  
 3   cargo          100 non-null    str  
 4   salario        90 non-null     str  
 5   data_admissao  100 non-null    str  
 6   email          92 non-null     str  
 7   ativo          100 non-null    str  
dtypes: int64(1), str(7)
memory usage: 6.4 KB


In [11]:
df.isnull()

,id,nome,departamento,cargo,salario,data_admissao,email,ativo
0,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...
95,False,False,False,False,False,False,False,False
96,False,False,False,False,False,False,False,False
97,False,False,False,False,False,False,False,False
98,False,False,False,False,False,False,False,False


---
## Problema 1 — Valores nulos
Identificamos quantos nulos existem por coluna antes de decidir o tratamento.

In [15]:
# Contagem de nulos por coluna
print('Nulos por coluna:')
nulos_tabular = df.isnull().sum()
print(nulos_tabular)


Nulos por coluna:
id                0
nome              0
departamento      0
cargo             0
salario          10
data_admissao     0
email             8
ativo             0
dtype: int64


In [14]:
total_nulos = df.isnull().sum().sum()
print(f'\nTotal de nulos: {total_nulos}')


Total de nulos: 18


### Estratégias de tratamento

**`salario` nulo** → preencheremos com a mediana (mais robusta que a média quando há outliers)  
**`email` nulo** → preencheremos com `'sem_email'` — valor padrão lógico para o contexto

In [16]:
# Salvar shape antes
shape_antes = df.shape[0]

# Preencher salario nulo com a mediana
mediana_salario = pd.to_numeric(df['salario'], errors='coerce').median()
df['salario'] = df['salario'].fillna(mediana_salario)

# Preencher email nulo com valor padrão
df['email'] = df['email'].fillna('sem_email')

print('Nulos restantes:')
print(df.isnull().sum())

Nulos restantes:
id               0
nome             0
departamento     0
cargo            0
salario          0
data_admissao    0
email            0
ativo            0
dtype: int64


---
## Problema 2 — Linhas duplicadas
O mesmo funcionário pode ter sido cadastrado mais de uma vez com IDs diferentes.

In [ ]:
# Verificar quantas duplicatas existem
print(f'Duplicatas encontradas: {df.duplicated().sum()}')

# Visualizar as duplicatas
display(df[df.duplicated(keep=False)].sort_values('nome').head(20))

In [ ]:
# Remover duplicatas — manter a primeira ocorrência
df = df.drop_duplicates()

print(f'Shape após remover duplicatas: {df.shape}')
print(f'Registros removidos: {shape_antes - df.shape[0]}')

---
## Problema 3 — Tipo de dado errado
A coluna `salario` chegou como texto (`object`) em vez de número. Causa: separador decimal com vírgula e aspas.

In [ ]:
# Ver o tipo atual
print('Tipo atual de salario:', df['salario'].dtype)

# Ver algumas amostras problemáticas
print('\nAmostras:')
print(df['salario'].head(15).tolist())

In [ ]:
# Passo 1: remover aspas e trocar vírgula por ponto
df['salario'] = df['salario'].astype(str).str.replace('"', '', regex=False)
df['salario'] = df['salario'].str.replace(',', '.', regex=False)

# Passo 2: converter para numérico
df['salario'] = pd.to_numeric(df['salario'], errors='coerce')

print('Tipo após conversão:', df['salario'].dtype)
print('Nulos gerados na conversão:', df['salario'].isnull().sum())

---
## Problema 4 — Datas inconsistentes
A coluna `data_admissao` tem três formatos diferentes: `YYYY-MM-DD`, `DD/MM/YYYY` e `Mon-YYYY`.

In [ ]:
# Ver os formatos presentes
print('Amostras de data_admissao:')
print(df['data_admissao'].head(20).tolist())

In [ ]:
# pd.to_datetime lida com múltiplos formatos automaticamente
df['data_admissao'] = pd.to_datetime(df['data_admissao'],
                                      errors='coerce',
                                      dayfirst=True)

print('Tipo após conversão:', df['data_admissao'].dtype)
print('Datas não convertidas (NaT):', df['data_admissao'].isnull().sum())

# Ver quais não foram convertidas
display(df[df['data_admissao'].isnull()])

---
## Problema 5 — Categorias sujas
A coluna `departamento` tem o mesmo valor escrito de muitas formas diferentes — isso quebra `groupby` e filtros.

In [ ]:
# Ver todos os valores únicos
print('Valores únicos em departamento:')
print(sorted(df['departamento'].unique()))

In [ ]:
# Passo 1: remover espaços e padronizar para maiúsculo
df['departamento'] = df['departamento'].str.strip().str.upper()

print('Após str.strip().str.upper():')
print(sorted(df['departamento'].unique()))

In [ ]:
# Passo 2: mapear todas as variações para o valor correto
mapa_dep = {
    'T.I.':                      'TI',
    'TECNOLOGIA':                 'TI',
    'TECNOLOGIA DA INFORMAÇÃO':   'TI',
    'TI ':                        'TI',
    'R.H.':                       'RH',
    'RECURSOS HUMANOS':            'RH',
    'RH ':                        'RH',
    'FIN.':                       'FINANCEIRO',
    'FINANCEIRO ':                'FINANCEIRO',
    'MKT':                        'MARKETING',
    'MARKETING ':                 'MARKETING',
    'VEND.':                      'VENDAS',
    'VENDAS ':                    'VENDAS',
    'OPER.':                      'OPERAÇÕES',
    'OPERACOES':                  'OPERAÇÕES',
    'OPERAÇÕES ':                 'OPERAÇÕES',
}

df['departamento'] = df['departamento'].map(mapa_dep).fillna(df['departamento'])

print('Valores únicos após mapeamento:')
print(sorted(df['departamento'].unique()))

---
## Problema 6 — Espaços invisíveis
Strings com espaços no início ou fim causam falhas silenciosas em filtros e agrupamentos.

In [ ]:
# Demonstrar o problema
print('Teste SEM strip:')
print(df[df['nome'] == 'Ana Lima'].shape[0], 'resultado(s)')

print('\nTeste COM strip direto na comparação:')
print(df[df['nome'].str.strip() == 'Ana Lima'].shape[0], 'resultado(s)')

In [ ]:
# Aplicar str.strip() em todas as colunas de texto
colunas_texto = ['nome', 'departamento', 'cargo', 'email', 'ativo']

for col in colunas_texto:
    df[col] = df[col].str.strip()

# Confirmar que o filtro agora funciona
print('Teste após str.strip() na coluna:')
print(df[df['nome'] == 'Ana Lima'].shape[0], 'resultado(s)')

---
## Problema 7 — Valores impossíveis
Dados que passaram pela validação do sistema mas não fazem sentido no mundo real.

In [ ]:
# Identificar salários negativos
print('Salários negativos:')
display(df[df['salario'] < 0][['nome', 'departamento', 'cargo', 'salario']])

In [ ]:
# Registrar shape antes
antes_impossiveis = df.shape[0]

# Remover salários negativos ou zerados
df = df[df['salario'] > 0]

print(f'Registros removidos: {antes_impossiveis - df.shape[0]}')
print(f'Shape atual: {df.shape}')

---
## Problema 8 — Outliers extremos
Valores absurdos que passaram pela validação — erros de digitação que distorcem qualquer análise.

In [ ]:
# Visualizar a distribuição do salário
print('Estatísticas do salário:')
print(df['salario'].describe())
print(f'\nMáximo: R$ {df["salario"].max():,.2f}')

In [ ]:
# Calcular limites com IQR
array_sal = np.array(df['salario'].dropna())

q1  = np.percentile(array_sal, 25)
q3  = np.percentile(array_sal, 75)
iqr = q3 - q1
lim_sup = q3 + (1.5 * iqr)
lim_inf = q1 - (1.5 * iqr)

print(f'Q1:              R$ {q1:,.2f}')
print(f'Q3:              R$ {q3:,.2f}')
print(f'IQR:             R$ {iqr:,.2f}')
print(f'Limite superior: R$ {lim_sup:,.2f}')
print(f'Limite inferior: R$ {lim_inf:,.2f}')

In [ ]:
# Identificar outliers superiores
print('Outliers superiores de salário:')
display(df[df['salario'] > lim_sup][['nome', 'departamento', 'cargo', 'salario']])

# Remover outliers extremos
antes_outlier = df.shape[0]
df = df[df['salario'] <= lim_sup]
print(f'\nRegistros removidos: {antes_outlier - df.shape[0]}')

---
## ✅ Resultado final — base limpa
Comparamos o estado inicial com o estado final após todos os tratamentos.

In [ ]:
print('=' * 45)
print('       RELATÓRIO DE AUDITORIA')
print('=' * 45)
print(f'Registros iniciais:       {shape_antes:>6}')
print(f'Registros finais:         {df.shape[0]:>6}')
print(f'Registros removidos:      {shape_antes - df.shape[0]:>6}')
print(f'Colunas:                  {df.shape[1]:>6}')
print('=' * 45)
print(f'Nulos restantes:          {df.isnull().sum().sum():>6}')
print(f'Duplicatas restantes:     {df.duplicated().sum():>6}')
print('=' * 45)

display(df.head(10))

In [ ]:
# Exportar base limpa
df.to_csv('funcionarios_naragroup_clean.csv', sep=';', index=False, encoding='utf-8')
print('✅ Base limpa exportada: funcionarios_naragroup_clean.csv')